# Etapa 3 — Análise e Dashboard

**Projeto:** TechPay — Monitoramento de Risco de Fraude

## Objetivo

Transformar as tabelas Gold em informações para a gestão de risco,
identificando padrões por canal, categoria, segmento e período. Os achados
serão traduzidos no formato Finding → Insight → Ação e apresentados em um
dashboard executivo.

In [14]:
from pathlib import Path
import sys

import pandas as pd
import duckdb
import plotly
import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("DuckDB:", duckdb.__version__)
print("Plotly:", plotly.__version__)

Python: 3.13.3
Pandas: 3.0.5
DuckDB: 1.5.5
Plotly: 7.0.0


In [15]:
pasta_atual = Path.cwd()

if pasta_atual.name == "etapa3_analise":
    PASTA_ETAPA3 = pasta_atual
    RAIZ_PROJETO = pasta_atual.parent.parent
else:
    RAIZ_PROJETO = pasta_atual
    PASTA_ETAPA3 = (
        RAIZ_PROJETO
        / "avaliacao_final"
        / "etapa3_analise"
    )

BANCO_DUCKDB = (
    RAIZ_PROJETO
    / "avaliacao_final"
    / "etapa2_bigdata"
    / "dados"
    / "techpay_avaliacao.duckdb"
)

ARQUIVO_DASHBOARD = PASTA_ETAPA3 / "dashboard.html"

print("Pasta da Etapa 3:", PASTA_ETAPA3)
print("Banco encontrado:", BANCO_DUCKDB.exists())
print("Banco DuckDB:", BANCO_DUCKDB)

Pasta da Etapa 3: c:\BigData\bigdata-curso-gabriel\avaliacao_final\etapa3_analise
Banco encontrado: True
Banco DuckDB: c:\BigData\bigdata-curso-gabriel\avaliacao_final\etapa2_bigdata\dados\techpay_avaliacao.duckdb


In [16]:
con = duckdb.connect(
    str(BANCO_DUCKDB),
    read_only=True
)

tabelas_disponiveis = con.execute("""
    SELECT
        table_schema,
        table_name
    FROM information_schema.tables
    WHERE table_schema IN (
        'raw',
        'bronze',
        'silver',
        'gold'
    )
    ORDER BY table_schema, table_name
""").df()

tabelas_disponiveis

,table_schema,table_name
0,bronze,transactions
1,gold,daily_metrics
2,gold,executive_summary
3,gold,fraud_by_category
4,gold,fraud_by_channel
5,gold,fraud_by_segment
6,raw,transactions
7,silver,transactions


In [17]:
resumo_executivo = con.execute("""
    SELECT *
    FROM gold.executive_summary
""").df()

resumo_executivo

,total_transactions,total_customers,total_amount,average_ticket,total_frauds,fraud_rate_pct,amount_at_risk
0,30000,7803,5207843.5,173.59,751.0,2.5033,119367.93


## Análise 1 — Risco por canal

Esta análise compara volume, quantidade de fraudes, participação nas fraudes,
taxa de fraude e índice de risco de cada canal. O índice de risco representa
quantas vezes a taxa do canal corresponde à taxa geral da base.

In [18]:
analise_canal = con.execute("""
    WITH taxa_geral AS (
        SELECT
            fraud_rate_pct AS taxa_geral_pct
        FROM gold.executive_summary
    )

    SELECT
        c.channel,
        c.total_transactions,
        c.total_frauds,

        ROUND(
            100.0 * c.total_frauds
            / SUM(c.total_frauds) OVER (),
            2
        ) AS participacao_fraudes_pct,

        c.fraud_rate_pct,

        ROUND(
            c.fraud_rate_pct / g.taxa_geral_pct,
            2
        ) AS indice_risco_vs_geral,

        c.amount_at_risk

    FROM gold.fraud_by_channel AS c
    CROSS JOIN taxa_geral AS g
    ORDER BY c.fraud_rate_pct DESC
""").df()

analise_canal

,channel,total_transactions,total_frauds,participacao_fraudes_pct,fraud_rate_pct,indice_risco_vs_geral,amount_at_risk
0,app,12021,455.0,60.59,3.7850,1.51,63909.91
1,atm,3007,54.0,7.19,1.7958,0.72,12395.30
2,web,9004,157.0,20.91,1.7437,0.70,31579.22
3,pos,5968,85.0,11.32,1.4243,0.57,11483.50


### Finding → Insight → Ação

**Finding:** o canal `app` registrou 455 fraudes, correspondentes a 60,59%
das ocorrências da base. Sua taxa de fraude foi de 3,7850%, equivalente a
1,51 vez a taxa geral, e o valor em risco alcançou R$ 63.909,91.

**Insight:** o aplicativo concentra uma parcela de fraudes
desproporcional ao seu volume de transações. Os demais canais apresentam
taxas inferiores à média geral, indicando que o risco adicional está
concentrado no comportamento das operações realizadas pelo aplicativo.

**Ação:** priorizar controles adicionais no canal `app`, como autenticação
em duas etapas ou validação complementar para operações de maior risco.
As regras devem combinar canal com horário, categoria do estabelecimento
e score de risco, evitando bloquear indiscriminadamente todas as
transações do aplicativo.

## Análise 2 — Risco por categoria de estabelecimento

Esta análise identifica quais categorias apresentam maior taxa de fraude e
qual participação representam no total de ocorrências. A taxa é comparada
com a média geral da TechPay.

In [19]:
analise_categoria = con.execute("""
    WITH taxa_geral AS (
        SELECT
            fraud_rate_pct AS taxa_geral_pct
        FROM gold.executive_summary
    )

    SELECT
        c.merchant_category,
        c.total_transactions,
        c.total_frauds,

        ROUND(
            100.0 * c.total_frauds
            / SUM(c.total_frauds) OVER (),
            2
        ) AS participacao_fraudes_pct,

        c.fraud_rate_pct,

        ROUND(
            c.fraud_rate_pct / g.taxa_geral_pct,
            2
        ) AS indice_risco_vs_geral,

        c.amount_at_risk

    FROM gold.fraud_by_category AS c
    CROSS JOIN taxa_geral AS g
    ORDER BY c.fraud_rate_pct DESC
""").df()

analise_categoria

,merchant_category,total_transactions,total_frauds,participacao_fraudes_pct,fraud_rate_pct,indice_risco_vs_geral,amount_at_risk
0,viagem,3010,160.0,21.30,5.3156,2.12,25966.40
1,saude,2951,73.0,9.72,2.4737,0.99,11141.53
2,alimentacao,5976,138.0,18.38,2.3092,0.92,21017.00
3,varejo,9047,203.0,27.03,2.2438,0.90,34870.54
4,servicos,4576,90.0,11.98,1.9668,0.79,14197.18
5,eletronico,4440,87.0,11.58,1.9595,0.78,12175.28


### Finding → Insight → Ação

**Finding:** a categoria `viagem` apresentou taxa de fraude de 5,3156%,
equivalente a 2,12 vezes a taxa geral. Foram registradas 160 fraudes,
21,30% do total, e R$ 25.966,40 em valor associado ao risco.

**Insight:** as operações de viagem possuem risco desproporcional ao seu
volume. Embora representem aproximadamente 10% das transações, concentram
mais de um quinto das fraudes, indicando que a categoria é uma dimensão
relevante para as regras de monitoramento.

**Ação:** aplicar monitoramento reforçado às transações de viagem,
principalmente quando combinadas com aplicativo, horário de madrugada,
segmento de maior risco ou score elevado. Operações com múltiplos sinais
de risco podem ser direcionadas para autenticação adicional ou análise
manual.

## Análise 3 — Combinação entre canal e categoria

A análise verifica se o risco aumenta quando duas dimensões são observadas
em conjunto. Isso permite criar regras mais específicas do que bloquear
isoladamente um canal ou uma categoria.

In [20]:
analise_combinacao = con.execute("""
    WITH combinacoes AS (
        SELECT
            channel,
            merchant_category,
            COUNT(*) AS total_transactions,
            SUM(fraud_indicator) AS total_frauds,
            ROUND(
                100.0 * AVG(fraud_indicator),
                4
            ) AS fraud_rate_pct,
            ROUND(
                SUM(
                    CASE
                        WHEN is_fraud THEN amount
                        ELSE 0
                    END
                ),
                2
            ) AS amount_at_risk
        FROM silver.transactions
        GROUP BY
            channel,
            merchant_category
    ),

    taxa_geral AS (
        SELECT
            fraud_rate_pct AS taxa_geral_pct
        FROM gold.executive_summary
    )

    SELECT
        c.channel,
        c.merchant_category,
        c.total_transactions,
        c.total_frauds,
        c.fraud_rate_pct,

        ROUND(
            c.fraud_rate_pct / g.taxa_geral_pct,
            2
        ) AS indice_risco_vs_geral,

        c.amount_at_risk

    FROM combinacoes AS c
    CROSS JOIN taxa_geral AS g

    WHERE c.total_transactions >= 200

    ORDER BY
        c.fraud_rate_pct DESC,
        c.total_transactions DESC

    LIMIT 10
""").df()

analise_combinacao

,channel,merchant_category,total_transactions,total_frauds,fraud_rate_pct,indice_risco_vs_geral,amount_at_risk
0,app,viagem,1237,99.0,8.0032,3.20,15578.67
1,app,alimentacao,2409,92.0,3.8190,1.53,12514.79
2,pos,viagem,612,23.0,3.7582,1.50,1947.62
3,atm,viagem,269,10.0,3.7175,1.49,2207.66
4,app,saude,1189,41.0,3.4483,1.38,4803.04
5,app,varejo,3600,117.0,3.2500,1.30,15924.61
6,web,viagem,892,28.0,3.1390,1.25,6232.45
7,app,eletronico,1777,55.0,3.0951,1.24,7829.57
8,app,servicos,1809,51.0,2.8192,1.13,7259.23
9,web,saude,849,21.0,2.4735,0.99,3629.63


### Finding → Insight → Ação

**Finding:** a combinação entre canal `app` e categoria `viagem`
apresentou taxa de fraude de 8,0032%, com 99 ocorrências e R$ 15.578,67
em risco. Essa taxa equivale a 3,20 vezes a média geral da TechPay.

**Insight:** canal e categoria não atuam de forma independente. A
combinação `app + viagem` produz um nível de risco maior do que o
observado isoladamente no aplicativo ou na categoria viagem. Portanto,
regras baseadas em apenas uma dimensão deixariam de capturar parte
importante do padrão.

**Ação:** criar uma regra combinada para transações de viagem realizadas
pelo aplicativo. A regra não precisa recusar automaticamente a operação,
mas pode solicitar autenticação adicional quando estiver acompanhada por
outros sinais, como madrugada, score elevado ou segmento High-Risk.

## Análise 4 — Risco por período do dia

Esta análise compara o comportamento das fraudes entre madrugada, manhã,
tarde e noite. Também detalha como o risco da combinação entre aplicativo
e viagem varia ao longo do dia.

In [21]:
analise_periodo = con.execute("""
    WITH taxa_geral AS (
        SELECT
            fraud_rate_pct AS taxa_geral_pct
        FROM gold.executive_summary
    )

    SELECT
        s.period_of_day,
        COUNT(*) AS total_transactions,
        SUM(s.fraud_indicator) AS total_frauds,

        ROUND(
            100.0 * AVG(s.fraud_indicator),
            4
        ) AS fraud_rate_pct,

        ROUND(
            100.0 * SUM(s.fraud_indicator)
            / SUM(SUM(s.fraud_indicator)) OVER (),
            2
        ) AS participacao_fraudes_pct,

        ROUND(
            (
                100.0 * AVG(s.fraud_indicator)
            ) / g.taxa_geral_pct,
            2
        ) AS indice_risco_vs_geral,

        ROUND(
            SUM(
                CASE
                    WHEN s.is_fraud THEN s.amount
                    ELSE 0
                END
            ),
            2
        ) AS amount_at_risk

    FROM silver.transactions AS s
    CROSS JOIN taxa_geral AS g

    GROUP BY
        s.period_of_day,
        g.taxa_geral_pct

    ORDER BY fraud_rate_pct DESC
""").df()

analise_periodo

,period_of_day,total_transactions,total_frauds,fraud_rate_pct,participacao_fraudes_pct,indice_risco_vs_geral,amount_at_risk
0,Madrugada,6291,218.0,3.4653,29.03,1.38,33995.50
1,Manhã,8776,201.0,2.2903,26.76,0.91,33098.30
2,Noite,7466,170.0,2.2770,22.64,0.91,25348.48
3,Tarde,7467,162.0,2.1695,21.57,0.87,26925.65


In [22]:
app_viagem_periodo = con.execute("""
    SELECT
        period_of_day,
        COUNT(*) AS total_transactions,
        SUM(fraud_indicator) AS total_frauds,

        ROUND(
            100.0 * AVG(fraud_indicator),
            4
        ) AS fraud_rate_pct,

        ROUND(
            SUM(
                CASE
                    WHEN is_fraud THEN amount
                    ELSE 0
                END
            ),
            2
        ) AS amount_at_risk

    FROM silver.transactions

    WHERE channel = 'app'
      AND merchant_category = 'viagem'

    GROUP BY period_of_day
    ORDER BY fraud_rate_pct DESC
""").df()

app_viagem_periodo

,period_of_day,total_transactions,total_frauds,fraud_rate_pct,amount_at_risk
0,Madrugada,249,23.0,9.2369,2216.11
1,Manhã,370,33.0,8.9189,5628.58
2,Noite,320,26.0,8.1250,4876.13
3,Tarde,298,17.0,5.7047,2857.85


### Finding → Insight → Ação

**Finding:** a madrugada apresentou taxa geral de fraude de 3,4653%,
equivalente a 1,38 vez a média da TechPay. Embora concentre cerca de 21%
das transações, respondeu por 29,03% das fraudes. Na combinação
`app + viagem`, a taxa da madrugada chegou a 9,2369%.

**Insight:** o horário amplia o risco já existente nas transações de
viagem realizadas pelo aplicativo. Entretanto, a combinação permanece
arriscada nos demais períodos, com taxas entre 5,7047% e 8,9189%.
Portanto, a madrugada deve funcionar como agravante, e não como único
critério de alerta.

**Ação:** elevar a prioridade dos alertas para transações `app + viagem`
realizadas entre 0h e 4h. Combinar esse sinal com valor, score de risco e
segmento do cliente, aplicando autenticação adicional ou análise manual
nos casos que acumularem múltiplos fatores.

## Análise 5 — Risco por segmento

Esta análise verifica como a taxa de fraude se distribui entre os segmentos
Premium, Standard e High-Risk, comparando cada grupo com a taxa geral.

In [23]:
analise_segmento = con.execute("""
    WITH taxa_geral AS (
        SELECT
            fraud_rate_pct AS taxa_geral_pct
        FROM gold.executive_summary
    )

    SELECT
        s.segment,
        s.total_transactions,
        s.total_customers,
        s.total_frauds,

        ROUND(
            100.0 * s.total_frauds
            / SUM(s.total_frauds) OVER (),
            2
        ) AS participacao_fraudes_pct,

        s.fraud_rate_pct,

        ROUND(
            s.fraud_rate_pct / g.taxa_geral_pct,
            2
        ) AS indice_risco_vs_geral,

        s.amount_at_risk

    FROM gold.fraud_by_segment AS s
    CROSS JOIN taxa_geral AS g
    ORDER BY s.fraud_rate_pct DESC
""").df()

analise_segmento

,segment,total_transactions,total_customers,total_frauds,participacao_fraudes_pct,fraud_rate_pct,indice_risco_vs_geral,amount_at_risk
0,High-Risk,2915,2475,282.0,37.55,9.6741,3.86,42156.04
1,Standard,10645,5902,312.0,41.54,2.9310,1.17,50874.90
2,Premium,16440,6940,157.0,20.91,0.9550,0.38,26336.99


### Finding → Insight → Ação

**Finding:** o segmento High-Risk apresentou taxa de fraude de 9,6741%,
equivalente a 3,86 vezes a taxa geral. O segmento respondeu por 37,55%
das fraudes, embora represente menos de 10% das transações.

**Insight:** a classificação do cliente possui forte capacidade de
diferenciação do risco. Entretanto, o segmento Standard registrou a maior
quantidade absoluta de fraudes, com 312 ocorrências, devido ao seu volume
maior de transações.

**Ação:** adotar políticas diferentes por segmento. Para clientes
High-Risk, utilizar limites e autenticação reforçada. Para o segmento
Standard, combinar segmento com canal, categoria, horário e score, pois o
volume elevado produz a maior quantidade absoluta de fraudes.

# Dashboard executivo

O dashboard consolida indicadores gerais, evolução temporal, composição
do risco e detalhamento das combinações prioritárias. Seu objetivo é
permitir que a gestão identifique rapidamente onde concentrar controles
antifraude.

In [24]:
# Dados utilizados no dashboard
daily_metrics = con.execute("""
    SELECT *
    FROM gold.daily_metrics
    ORDER BY transaction_date
""").df()

daily_metrics["transaction_date"] = pd.to_datetime(
    daily_metrics["transaction_date"]
)

daily_metrics["moving_average_7d"] = (
    daily_metrics["total_frauds"]
    .rolling(window=7, min_periods=1)
    .mean()
)

resumo = resumo_executivo.iloc[0]

# Formatação brasileira para a tabela
def inteiro_br(valor):
    return f"{int(valor):,}".replace(",", ".")

def decimal_br(valor, casas=2):
    texto = f"{float(valor):,.{casas}f}"
    return texto.replace(",", "X").replace(".", ",").replace("X", ".")

detalhe_dashboard = analise_combinacao.copy()

fig = make_subplots(
    rows=4,
    cols=4,
    row_heights=[0.14, 0.32, 0.28, 0.26],
    vertical_spacing=0.075,
    horizontal_spacing=0.06,
    specs=[
        [
            {"type": "indicator"},
            {"type": "indicator"},
            {"type": "indicator"},
            {"type": "indicator"},
        ],
        [
            {"type": "xy", "colspan": 4},
            None,
            None,
            None,
        ],
        [
            {"type": "xy", "colspan": 2},
            None,
            {"type": "xy", "colspan": 2},
            None,
        ],
        [
            {"type": "table", "colspan": 4},
            None,
            None,
            None,
        ],
    ],
    subplot_titles=(
        "",
        "",
        "",
        "",
        "Evolução diária das fraudes",
        "Taxa de fraude por canal",
        "Taxa de fraude por categoria",
        "Combinações prioritárias de canal e categoria",
    ),
)

# KPI 1
fig.add_trace(
    go.Indicator(
        mode="number",
        value=resumo["total_transactions"],
        title={"text": "Total de transações"},
        number={"valueformat": ",.0f"},
    ),
    row=1,
    col=1,
)

# KPI 2
fig.add_trace(
    go.Indicator(
        mode="number",
        value=resumo["total_frauds"],
        title={"text": "Fraudes identificadas"},
        number={
            "valueformat": ",.0f",
            "font": {"color": "#D64545"},
        },
    ),
    row=1,
    col=2,
)

# KPI 3
fig.add_trace(
    go.Indicator(
        mode="number",
        value=resumo["fraud_rate_pct"],
        title={"text": "Taxa geral de fraude"},
        number={
            "valueformat": ".2f",
            "suffix": "%",
            "font": {"color": "#F2A400"},
        },
    ),
    row=1,
    col=3,
)

# KPI 4
fig.add_trace(
    go.Indicator(
        mode="number",
        value=resumo["amount_at_risk"],
        title={"text": "Valor em risco"},
        number={
            "valueformat": ",.2f",
            "prefix": "R$ ",
            "font": {"color": "#6F42C1"},
        },
    ),
    row=1,
    col=4,
)

# Tendência diária
fig.add_trace(
    go.Scatter(
        x=daily_metrics["transaction_date"],
        y=daily_metrics["total_frauds"],
        mode="lines",
        name="Fraudes diárias",
        line={"color": "rgba(214,69,69,0.35)", "width": 1},
        hovertemplate="%{x|%d/%m/%Y}<br>Fraudes: %{y}<extra></extra>",
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=daily_metrics["transaction_date"],
        y=daily_metrics["moving_average_7d"],
        mode="lines",
        name="Média móvel de 7 dias",
        line={"color": "#D64545", "width": 3},
        hovertemplate="%{x|%d/%m/%Y}<br>Média: %{y:.2f}<extra></extra>",
    ),
    row=2,
    col=1,
)

# Barras por canal
cores_canal = {
    "app": "#D64545",
    "atm": "#F2A400",
    "web": "#2F80ED",
    "pos": "#27AE60",
}

fig.add_trace(
    go.Bar(
        x=analise_canal["channel"],
        y=analise_canal["fraud_rate_pct"],
        text=analise_canal["fraud_rate_pct"].map(
            lambda x: f"{x:.2f}%"
        ),
        textposition="outside",
        marker_color=[
            cores_canal.get(canal, "#2F80ED")
            for canal in analise_canal["channel"]
        ],
        name="Canal",
        showlegend=False,
        hovertemplate=(
            "Canal: %{x}<br>"
            "Taxa: %{y:.2f}%<extra></extra>"
        ),
    ),
    row=3,
    col=1,
)

# Barras por categoria
fig.add_trace(
    go.Bar(
        x=analise_categoria["merchant_category"],
        y=analise_categoria["fraud_rate_pct"],
        text=analise_categoria["fraud_rate_pct"].map(
            lambda x: f"{x:.2f}%"
        ),
        textposition="outside",
        marker_color=[
            "#D64545" if categoria == "viagem"
            else "#2F80ED"
            for categoria in analise_categoria["merchant_category"]
        ],
        name="Categoria",
        showlegend=False,
        hovertemplate=(
            "Categoria: %{x}<br>"
            "Taxa: %{y:.2f}%<extra></extra>"
        ),
    ),
    row=3,
    col=3,
)

# Tabela de detalhes
fig.add_trace(
    go.Table(
        header={
            "values": [
                "<b>Canal</b>",
                "<b>Categoria</b>",
                "<b>Transações</b>",
                "<b>Fraudes</b>",
                "<b>Taxa</b>",
                "<b>Índice de risco</b>",
                "<b>Valor em risco</b>",
            ],
            "fill_color": "#17324D",
            "font": {"color": "white", "size": 12},
            "align": "center",
            "height": 30,
        },
        cells={
            "values": [
                detalhe_dashboard["channel"],
                detalhe_dashboard["merchant_category"],
                detalhe_dashboard["total_transactions"].map(inteiro_br),
                detalhe_dashboard["total_frauds"].map(inteiro_br),
                detalhe_dashboard["fraud_rate_pct"].map(
                    lambda x: f"{decimal_br(x)}%"
                ),
                detalhe_dashboard["indice_risco_vs_geral"].map(
                    lambda x: f"{decimal_br(x)}x"
                ),
                detalhe_dashboard["amount_at_risk"].map(
                    lambda x: f"R$ {decimal_br(x)}"
                ),
            ],
            "fill_color": [
                ["#F5F7FA", "white"] * 5
            ],
            "align": "center",
            "height": 26,
            "font": {"color": "#243447", "size": 11},
        },
    ),
    row=4,
    col=1,
)

fig.update_yaxes(
    title_text="Quantidade de fraudes",
    row=2,
    col=1,
)

fig.update_yaxes(
    title_text="Taxa de fraude (%)",
    rangemode="tozero",
    row=3,
    col=1,
)

fig.update_yaxes(
    title_text="Taxa de fraude (%)",
    rangemode="tozero",
    row=3,
    col=3,
)

fig.update_xaxes(
    title_text="Data",
    row=2,
    col=1,
)

fig.update_layout(
    title={
        "text": (
            "<b>TechPay — Monitoramento de Risco de Fraude</b>"
            "<br><sup>Visão executiva das transações de 2025</sup>"
        ),
        "x": 0.5,
        "xanchor": "center",
    },
    height=1550,
    template="plotly_white",
    margin={"l": 55, "r": 55, "t": 120, "b": 40},
    separators=",.",
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 0.665,
        "xanchor": "center",
        "x": 0.5,
    },
    font={
        "family": "Arial",
        "color": "#243447",
    },
)

fig.show()

In [25]:
fig.write_html(
    str(ARQUIVO_DASHBOARD),
    include_plotlyjs=True,
    full_html=True,
)

print("Dashboard criado em:", ARQUIVO_DASHBOARD)
print("Arquivo encontrado:", ARQUIVO_DASHBOARD.exists())

Dashboard criado em: c:\BigData\bigdata-curso-gabriel\avaliacao_final\etapa3_analise\dashboard.html
Arquivo encontrado: True


In [26]:
con.close()
print("Análises concluídas e conexão encerrada.")

Análises concluídas e conexão encerrada.
